In [1]:
from huggingface_hub import login

login("")

In [2]:
from datasets import load_dataset
dataset = load_dataset("uitnlp/vigoemotions")

README.md: 0.00B [00:00, ?B/s]

train.csv: 0.00B [00:00, ?B/s]

val.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/16531 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2066 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2067 [00:00<?, ? examples/s]

In [3]:
import re

NUM_LABELS = 28  # ViGoEmotions

def parse_labels(label_str):
    indices = list(map(int, re.findall(r"\d+", label_str)))
    multi_hot = [0.0] * NUM_LABELS
    for idx in indices:
        multi_hot[idx] = 1.0
    return multi_hot

def preprocess(example):
    example["labels"] = parse_labels(example["labels"])
    return example

dataset = dataset.map(preprocess)

Map:   0%|          | 0/16531 [00:00<?, ? examples/s]

Map:   0%|          | 0/2066 [00:00<?, ? examples/s]

Map:   0%|          | 0/2067 [00:00<?, ? examples/s]

In [4]:
import pandas as pd
import numpy as np

def evaluate_from_csv(csv_path, dataset):
    df = pd.read_csv(csv_path)

    # split dev/test (skip header already handled by pandas)
    dev_df = df.iloc[:2066]
    print(dev_df)
    test_df = df.iloc[2066:]

    def process_split(split_df, split_name):
        results = []

        # build id -> index mapping for dataset split
        id2idx = {id_: i for i, id_ in enumerate(dataset[split_name]["id"])}
        train_id2idx = {id_: i for i, id_ in enumerate(dataset["train"]["id"])}

        for _, row in split_df.iterrows():
            query_id = row[0]
            similar_ids = eval(row[1])   # list of 5 ids
            opposite_ids = eval(row[2])  # list of 5 ids

            # get query labels
            q_idx = id2idx[query_id]
            query_labels = np.array(dataset[split_name]["labels"][q_idx])

            # get similar labels and average
            sim_labels = np.array([
                dataset["train"]["labels"][train_id2idx[sid]]
                for sid in similar_ids
            ])
            sim_avg = sim_labels.mean(axis=0)

            # get opposite labels and average
            opp_labels = np.array([
                dataset["train"]["labels"][train_id2idx[oid]]
                for oid in opposite_ids
            ])
            opp_avg = opp_labels.mean(axis=0)

            # prediction: similar - opposite
            pred = sim_avg - opp_avg

            results.append({
                "query_id": query_id,
                "query_labels": query_labels,
                "prediction": pred
            })

        return results

    dev_results = process_split(dev_df, "validation")
    test_results = process_split(test_df, "test")

    return dev_results, test_results

In [5]:
dev_res1, test_res1 = evaluate_from_csv('/kaggle/input/datasets/nbmdat/nlp-results/retrieval_results.csv',dataset)

       query_id                                        similar_ids  \
0         13119    ['23081', '3097', 'tik005495', '2962', '18525']   
1          2525         ['544', '27248', '14708', '31442', '5747']   
2       x001179  ['tik013499', 'x001045', 'red000013', 'x000040...   
3         17079  ['17080', '16066', 'you002997', 'you000658', '...   
4         29305           ['6797', '381', '5330', '15524', '4007']   
...         ...                                                ...   
2061  tik010671  ['you002846', '4110', 'tik003465', '17075', 'y...   
2062       4179   ['31466', 'tik011844', '5008', '31411', '13135']   
2063  tik018299  ['tik014501', 'tik017736', 'tik019068', 'tik01...   
2064  you001924  ['you005594', 'you001322', 'x000554', 'x000880...   
2065       6403         ['12044', '5475', '1461', '6366', '30364']   

                                          different_ids  
0     ['tik022795', 'tik015151', '14641', 'tik021839...  
1     ['you002020', 'you001609', 'you002078

/tmp/ipykernel_16/2274719756.py:20: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  query_id = row[0]
/tmp/ipykernel_16/2274719756.py:21: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  similar_ids = eval(row[1])   # list of 5 ids
/tmp/ipykernel_16/2274719756.py:22: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  opposite_ids = eval(row[2])  # list of 5 ids


In [6]:
from sklearn.metrics import f1_score

def compute_optimal_thresholds(dev_results, num_classes, num_steps=101):
    """
    dev_results: list of dicts with keys:
        - "query_labels": (C,)
        - "prediction": (C,)
    num_classes: number of label dimensions
    num_steps: number of threshold candidates to scan per class

    Returns:
        thresholds: np.array of shape (C,)
    """

    # stack arrays
    y_true = np.stack([r["query_labels"] for r in dev_results])  # (N, C)
    y_pred = np.stack([r["prediction"] for r in dev_results])    # (N, C)

    thresholds = np.zeros(num_classes)

    for c in range(num_classes):
        best_thr = 0.0
        best_f1 = -1.0

        # define search range based on prediction distribution
        min_val = y_pred[:, c].min()
        max_val = y_pred[:, c].max()
        candidates = np.linspace(min_val, max_val, num_steps)

        for thr in candidates:
            y_bin = (y_pred[:, c] >= thr).astype(int)
            f1 = f1_score(y_true[:, c], y_bin, zero_division=0)

            if f1 > best_f1:
                best_f1 = f1
                best_thr = thr

        thresholds[c] = best_thr

    return thresholds

In [7]:
thresholds1 = compute_optimal_thresholds(dev_res1,NUM_LABELS)
print(thresholds1)

[0.608 0.004 0.208 0.604 0.21  0.406 0.412 0.4   0.208 0.6   0.208 0.22
 0.004 0.208 0.4   0.208 0.4   0.2   0.4   0.6   0.208 0.412 0.208 0.22
 0.22  0.22  0.006 0.202]


In [8]:
from sklearn.metrics import precision_score, recall_score

def evaluate_with_thresholds(test_results, thresholds):
    """
    test_results: list of dicts with keys:
        - "query_labels": (C,)
        - "prediction": (C,)
    thresholds: np.array of shape (C,)

    Returns:
        dict of evaluation metrics
    """

    # stack arrays
    y_true = np.stack([r["query_labels"] for r in test_results])  # (N, C)
    y_pred = np.stack([r["prediction"] for r in test_results])    # (N, C)

    # apply per-class thresholds
    y_bin = (y_pred >= thresholds).astype(int)

    results = {
        #"precision_micro": precision_score(y_true, y_bin, average="micro", zero_division=0),
        "precision_macro": precision_score(y_true, y_bin, average="macro", zero_division=0),
        #"precision_weighted": precision_score(y_true, y_bin, average="weighted", zero_division=0),

        #"recall_micro": recall_score(y_true, y_bin, average="micro", zero_division=0),
        "recall_macro": recall_score(y_true, y_bin, average="macro", zero_division=0),
        #"recall_weighted": recall_score(y_true, y_bin, average="weighted", zero_division=0),

        "f1_micro": f1_score(y_true, y_bin, average="micro", zero_division=0),
        "f1_macro": f1_score(y_true, y_bin, average="macro", zero_division=0),
        "f1_weighted": f1_score(y_true, y_bin, average="weighted", zero_division=0),
    }

    return y_true, y_bin, results

In [9]:
true_labels1, pred_labels1 ,score1 = evaluate_with_thresholds(test_res1,thresholds1)

In [10]:
print(score1)

{'precision_macro': 0.4721448541990351, 'recall_macro': 0.5055008929753207, 'f1_micro': 0.478134110787172, 'f1_macro': 0.4699274742174842, 'f1_weighted': 0.5057155878027999}


This is the result when using constrastive learning fine-tuned ViSoBERT as a retriever. Retrieve 5 most similar and opposite examples, adding the average of similars and subtracting average of opposite, finally thresholding with threshold from dev set to receive multi-hot output.

In [11]:
df_out1 = pd.DataFrame({
        "true_labels": [list(row) for row in true_labels1],
        "predicted_labels": [list(row) for row in pred_labels1]
    })

df_out1.to_csv('/kaggle/working/retrieval_knn_based_preds.csv', index=False)

In [12]:
def process_new_format(csv_path, dataset):
    """
    Returns:
        y_true_pred: list of true labels (for threshold computation)
        y_pred: list of predicted labels (from similar examples)
        y_true_anchor: list of true labels (for anchor comparison)
        anchor_lists: list of lists containing anchor indices
    """

    df = pd.read_csv(csv_path)

    # build id -> index mappings
    val_id2idx = {id_: i for i, id_ in enumerate(dataset["validation"]["id"])}
    train_id2idx = {id_: i for i, id_ in enumerate(dataset["train"]["id"])}

    y_true_pred = []
    y_pred = []
    y_true_anchor = []
    anchor_lists = []

    for _, row in df.iterrows():
        query_id = row[0]

        # column 2: list of (id, value) -> extract ids
        sim_list = eval(row[1])  # [('id', value), ...]
        similar_ids = [sid for sid, _ in sim_list]

        # column 3: dict-like or list of tuples -> extract anchor indices
        anchor_obj = eval(row[2])
        if isinstance(anchor_obj, dict):
            anchor_keys = anchor_obj.keys()
        else:  # list of tuples
            anchor_keys = [k for k, _ in anchor_obj]

        anchor_indices = sorted(int(k.split("_")[1]) for k in anchor_keys)

        # true labels
        q_idx = val_id2idx[query_id]
        true_labels = np.array(dataset["validation"]["labels"][q_idx])

        # predicted labels = average over similar examples
        sim_labels = np.array([
            dataset["train"]["labels"][train_id2idx[sid]]
            for sid in similar_ids
        ])
        pred = sim_labels.mean(axis=0)

        # collect outputs
        y_true_pred.append(true_labels)
        y_pred.append(pred)

        #y_true_anchor.append(true_labels)
        anchor_lists.append(anchor_indices)

    return y_true_pred, y_pred, anchor_lists

In [13]:
true_labels2_dev , pred_labels2_dev, anchor_lists_dev = process_new_format('/kaggle/input/datasets/nbmdat/nlp-results/retrieval_results_emoembed_dev.csv', dataset)

/tmp/ipykernel_16/1461428133.py:22: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  query_id = row[0]
/tmp/ipykernel_16/1461428133.py:25: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  sim_list = eval(row[1])  # [('id', value), ...]
/tmp/ipykernel_16/1461428133.py:29: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  anchor_obj = eval(row[2])


In [14]:
import numpy as np
from sklearn.metrics import f1_score

def compute_optimal_thresholds_from_lists(y_true_pred, y_pred, num_steps=101):
    """
    y_true_pred: list/array of shape (N, C)
    y_pred: list/array of shape (N, C)

    Returns:
        thresholds: np.array of shape (C,)
    """

    y_true = np.array(y_true_pred)
    y_pred = np.array(y_pred)

    num_classes = y_true.shape[1]
    thresholds = np.zeros(num_classes)

    for c in range(num_classes):
        best_thr = 0.0
        best_f1 = -1.0

        min_val = y_pred[:, c].min()
        max_val = y_pred[:, c].max()
        candidates = np.linspace(min_val, max_val, num_steps)

        for thr in candidates:
            y_bin = (y_pred[:, c] >= thr).astype(int)
            f1 = f1_score(y_true[:, c], y_bin, zero_division=0)

            if f1 > best_f1:
                best_f1 = f1
                best_thr = thr

        thresholds[c] = best_thr

    return thresholds

In [15]:
thresholds2 = compute_optimal_thresholds_from_lists(true_labels2_dev,pred_labels2_dev)
print(thresholds2)

[0.21  0.006 0.21  0.208 0.21  0.21  0.21  0.208 0.008 0.41  0.208 0.01
 0.006 0.008 0.21  0.208 0.008 0.006 0.208 0.21  0.008 0.21  0.21  0.208
 0.21  0.01  0.01  0.008]


In [16]:
def compute_best_k_from_anchors(y_true_pred, anchor_lists, num_classes, max_k=5):
    """
    y_true_pred: list/array of shape (N, C)
    anchor_lists: list of lists, each containing ordered anchor indices
    num_classes: total number of labels (C)
    max_k: maximum k to try (default 5)

    Returns:
        best_k: int
        best_macro_f1: float
    """

    y_true = np.array(y_true_pred)

    best_k = 1
    best_f1 = -1.0

    for k in range(1, max_k + 1):
        y_pred_bin = []

        for anchors in anchor_lists:
            vec = np.zeros(num_classes, dtype=int)

            # take first k anchors (or fewer if not enough)
            selected = anchors[:k]

            for idx in selected:
                if 0 <= idx < num_classes:
                    vec[idx] = 1

            y_pred_bin.append(vec)

        y_pred_bin = np.array(y_pred_bin)

        macro_f1 = f1_score(y_true, y_pred_bin, average="macro", zero_division=0)

        if macro_f1 > best_f1:
            best_f1 = macro_f1
            best_k = k

    return best_k

In [17]:
thresholds3 = compute_best_k_from_anchors(true_labels2_dev,anchor_lists_dev,NUM_LABELS)

In [18]:
print(thresholds3)

5


In [19]:
def evaluate_test_with_two_thresholds(csv_path, dataset, thresholds, best_k):
    """
    csv_path: test file (same format as previous)
    dataset: HuggingFace dataset with 'test' and 'train'
    thresholds: np.array (C,) from similarity-based approach
    best_k: int from anchor-based approach

    Returns:
        dict with metrics for both methods
    """

    df = pd.read_csv(csv_path)

    # mappings
    test_id2idx = {id_: i for i, id_ in enumerate(dataset["test"]["id"])}
    train_id2idx = {id_: i for i, id_ in enumerate(dataset["train"]["id"])}

    y_true = []
    y_pred_sim = []
    y_pred_anchor = []

    for _, row in df.iterrows():
        query_id = row[0]

        # similar list
        sim_list = eval(row[1])  # [('id', value), ...]
        similar_ids = [sid for sid, _ in sim_list]

        # anchor list
        anchor_obj = eval(row[2])
        if isinstance(anchor_obj, dict):
            anchor_keys = anchor_obj.keys()
        else:
            anchor_keys = [k for k, _ in anchor_obj]

        anchor_indices = [int(k.split("_")[1]) for k in anchor_keys]

        # true labels
        q_idx = test_id2idx[query_id]
        true_labels = np.array(dataset["test"]["labels"][q_idx])
        y_true.append(true_labels)

        # --- similarity-based prediction ---
        sim_labels = np.array([
            dataset["train"]["labels"][train_id2idx[sid]]
            for sid in similar_ids
        ])
        pred_sim = sim_labels.mean(axis=0)
        y_pred_sim.append(pred_sim)

        # --- anchor-based prediction ---
        vec = np.zeros_like(true_labels)
        for idx in anchor_indices[:best_k]:
            if 0 <= idx < len(vec):
                vec[idx] = 1
        y_pred_anchor.append(vec)

    y_true = np.array(y_true)
    y_pred_sim = np.array(y_pred_sim)
    y_pred_anchor = np.array(y_pred_anchor)

    # apply thresholds for similarity-based
    y_bin_sim = (y_pred_sim >= thresholds).astype(int)

    def compute_metrics(y_t, y_p):
        return {
            #"precision_micro": precision_score(y_t, y_p, average="micro", zero_division=0),
            "precision_macro": precision_score(y_t, y_p, average="macro", zero_division=0),
            #"precision_weighted": precision_score(y_t, y_p, average="weighted", zero_division=0),
            #"recall_micro": recall_score(y_t, y_p, average="micro", zero_division=0),
            "recall_macro": recall_score(y_t, y_p, average="macro", zero_division=0),
            #"recall_weighted": recall_score(y_t, y_p, average="weighted", zero_division=0),
            "f1_micro": f1_score(y_t, y_p, average="micro", zero_division=0),
            "f1_macro": f1_score(y_t, y_p, average="macro", zero_division=0),
            "f1_weighted": f1_score(y_t, y_p, average="weighted", zero_division=0),
        }

    results = {
        "similarity_based": compute_metrics(y_true, y_bin_sim),
        "anchor_based": compute_metrics(y_true, y_pred_anchor),
    }

    return y_true, y_bin_sim, y_pred_anchor, results

In [20]:
true_labels23, pred_labels2, pred_labels3, results = evaluate_test_with_two_thresholds(
    '/kaggle/input/datasets/nbmdat/nlp-results/retrieval_results_emoembed_test.csv',
    dataset,
    thresholds2,
    thresholds3)

/tmp/ipykernel_16/375930981.py:23: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  query_id = row[0]
/tmp/ipykernel_16/375930981.py:26: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  sim_list = eval(row[1])  # [('id', value), ...]
/tmp/ipykernel_16/375930981.py:30: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  anchor_obj = eval(row[2])


In [21]:
print(results['similarity_based'])

{'precision_macro': 0.21986978047573108, 'recall_macro': 0.39123955545271644, 'f1_micro': 0.2953338236495873, 'f1_macro': 0.26132613033177793, 'f1_weighted': 0.31765941601455444}


Score of retrieval_emoembed_knn_based_preds

In [22]:
print(results['anchor_based'])

{'precision_macro': 0.12447116449019248, 'recall_macro': 0.3454705450633589, 'f1_micro': 0.2102682636408209, 'f1_macro': 0.1552201034499901, 'f1_weighted': 0.20028237931685597}


retrieval_emoembed_knanchor_based_preds

In [23]:
df_out2 = pd.DataFrame({
        "true_labels": [list(row) for row in true_labels23],
        "predicted_labels": [list(row) for row in pred_labels2]
    })

df_out2.to_csv('/kaggle/working/retrieval_emoembed_knn_based_preds.csv', index=False)

In [24]:
df_out3 = pd.DataFrame({
        "true_labels": [list(row) for row in true_labels23],
        "predicted_labels": [list(row) for row in pred_labels3]
    })

df_out3.to_csv('/kaggle/working/retrieval_emoembed_knanchor_based_preds.csv', index=False)

In [25]:
def evaluate_from_multihot_xlsx(xlsx_path):
    """
    xlsx file format:
        - column 3: true labels (multi-hot vector as string/list)
        - column 4: predicted labels (multi-hot vector as string/list)
    """

    df = pd.read_excel(xlsx_path, engine="openpyxl")

    def parse_vec(x):
        if isinstance(x, str):
            return np.array(eval(x))
        return np.array(x)

    y_true = np.stack(df.iloc[:, 2].apply(parse_vec).values)
    y_pred = np.stack(df.iloc[:, 3].apply(parse_vec).values)

    results = {
        #"precision_micro": precision_score(y_true, y_pred, average="micro", zero_division=0),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        #"precision_weighted": precision_score(y_true, y_pred, average="weighted", zero_division=0),

        #"recall_micro": recall_score(y_true, y_pred, average="micro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        #"recall_weighted": recall_score(y_true, y_pred, average="weighted", zero_division=0),

        "f1_micro": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }

    return y_true,y_pred, results

In [26]:
true_labels4, pred_labels4,score4 = evaluate_from_multihot_xlsx('/kaggle/input/datasets/nbmdat/nlp-results/multihot_output_RAG_preprocessyes.xlsx')
print(score4)

{'precision_macro': 0.4007091913901529, 'recall_macro': 0.3517372661226788, 'f1_micro': 0.3511111111111111, 'f1_macro': 0.31368553831758206, 'f1_weighted': 0.3602082548458446}


In [27]:
def compute_mislabel_matrix(y_true, y_pred):
    """
    Computes a mislabel co-occurrence matrix S of shape (C, C)

    S[i, j] counts samples where:
        - true label includes class i AND excludes class j
        - predicted label includes class j AND excludes class i

    Args:
        y_true: (N, C) binary multi-hot array
        y_pred: (N, C) binary multi-hot array

    Returns:
        S: (C, C) integer matrix
    """

    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)

    N, C = y_true.shape
    S = np.zeros((C, C), dtype=int)

    for i in range(C):
        for j in range(C):
            if i == j:
                continue

            mask = (
                (y_true[:, i] == 1) &
                (y_true[:, j] == 0) &
                (y_pred[:, j] == 1) &
                (y_pred[:, i] == 0)
            )

            S[i, j] = np.sum(mask)

    return S

In [28]:
mislabel_matrix1 = compute_mislabel_matrix(true_labels1, pred_labels1)
pd_matrix1 = pd.DataFrame(mislabel_matrix1)
pd_matrix1.to_csv('/kaggle/working/mislabel_matrix_retrieval_knn_based.csv', index=True)

In [29]:
mislabel_matrix2 = compute_mislabel_matrix(true_labels23, pred_labels2)
pd_matrix2 = pd.DataFrame(mislabel_matrix2)
pd_matrix2.to_csv('/kaggle/working/mislabel_matrix_retrieval_emoembed_knn_based.csv', index=True)

In [30]:
mislabel_matrix3 = compute_mislabel_matrix(true_labels23, pred_labels3)
pd_matrix3 = pd.DataFrame(mislabel_matrix3)
pd_matrix3.to_csv('/kaggle/working/mislabel_matrix_retrieval_emoembed_knnanchor_based.csv', index=True)

In [31]:
mislabel_matrix4 = compute_mislabel_matrix(true_labels4, pred_labels4)
pd_matrix4 = pd.DataFrame(mislabel_matrix4)
pd_matrix4.to_csv('/kaggle/working/mislabel_multihot_output_RAG.csv', index=True)

In [32]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/nbmdat/nlp-results/retrieval_results_emoembed_dev.csv
/kaggle/input/datasets/nbmdat/nlp-results/retrieval_results_emoembed_test.csv
/kaggle/input/datasets/nbmdat/nlp-results/anchors.csv
/kaggle/input/datasets/nbmdat/nlp-results/retrieval_results.csv
/kaggle/input/datasets/nbmdat/nlp-results/predictions_LoRA_preprocessyes.csv
/kaggle/input/datasets/nbmdat/nlp-results/multihot_output_RAG_preprocessyes.xlsx
